# 55 — OpenRouter Setup
**Goal:** Configure OpenRouter API, select models, manage costs and rate limits.

## 1. Why OpenRouter?

In [ ]:
print('''OpenRouter provides unified API to 200+ LLMs:
- Single endpoint for GPT-4, Claude, Gemini, Llama, Mistral, etc.
- Built-in fallback if a model is rate-limited or down
- Cost tracking per model
- No vendor lock-in

For resume analysis:
  - GPT-4-mini: Best for rewriting, generation
  - Claude 3 Haiku: Fast, cheap for classification
  - Llama 3 70B: Open-source alternative
  - Mistral Small: Budget option for simple tasks''')

## 2. Setting Up the Client

In [ ]:
# pip install openai
from openai import OpenAI
import os

# Get API key from environment
api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
if not api_key:
    api_key = ""  # Fill in your key here
    print("⚠ No API key found. Set OPENROUTER_API_KEY in .env or enter directly.")
    print("  Get one at: https://openrouter.ai/keys")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

# Test the connection
try:
    response = client.models.list()
    model_ids = [m.id for m in response.data if "free" in m.id][:5]
    print(f"✓ API connected. Free models available: {model_ids}")
except Exception as e:
    print(f"Connection test: {e}")
    print("(Expected if no API key — notebook works as reference.)")

## 3. Model Selection Helper

In [ ]:
MODEL_TIERS = {
    "cheap": ["mistralai/mistral-small-24b-instruct-2501", "google/gemini-2.0-flash-lite-001"],
    "balanced": ["openai/gpt-4o-mini", "anthropic/claude-3.5-haiku"],
    "best": ["openai/gpt-4o", "anthropic/claude-3.5-sonnet"],
}

def pick_model(task, quality="balanced"):
    """Pick the right model for the task."""
    task_recommendations = {
        "classification": "cheap",
        "extraction": "balanced",
        "rewriting": "balanced",
        "generation": "best",
    }
    tier = task_recommendations.get(task, quality)
    return MODEL_TIERS[tier][0]

for task in ["classification", "extraction", "rewriting", "generation"]:
    print(f"  {task:15s} -> {pick_model(task)}")

## 4. Cost Tracking

In [ ]:
COST_PER_1K = {
    "openai/gpt-4o-mini": {"input": 0.000150, "output": 0.000600},
    "anthropic/claude-3.5-haiku": {"input": 0.000250, "output": 0.001250},
    "mistralai/mistral-small-24b": {"input": 0.000200, "output": 0.000600},
}

def estimate_cost(model, input_tokens, output_tokens):
    costs = COST_PER_1K.get(model)
    if not costs:
        return "Check pricing page"
    cost = (input_tokens / 1000 * costs["input"]) + (output_tokens / 1000 * costs["output"])
    return f"${cost:.6f}"

# Estimate for a typical resume
print("Cost estimate (resume rewrite, ~500 in / ~300 out):")
for model in COST_PER_1K:
    print(f"  {model:35s}: {estimate_cost(model, 500, 300)}")

## Summary: OpenRouter gives access to all major models through one API. Pick model by task complexity.